# Lab 1 – PCA · Dataset 2: Wine Quality (UCI ID 186)

Files: `winequality-red.csv`, `winequality-white.csv`. Separator is `;`.

In [69]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

## I. Study the dataset
Each file is loaded separately. Cleaning is done per file.

In [70]:
DATA_DIR = "data"
red = pd.read_csv(f"{DATA_DIR}/winequality-red.csv", sep=";")
white = pd.read_csv(f"{DATA_DIR}/winequality-white.csv", sep=";")

datasets = {"red": red, "white": white}

for name, df in datasets.items():
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} columns")

# Sanity check: 11 features + quality = 12 columns.
# If you see 1 column, the separator is wrong.
assert red.shape[1] == 12 and white.shape[1] == 12, "Wrong separator? Use sep=';'"
assert list(red.columns) == list(white.columns), "Column names differ between files"

red: 1599 rows, 12 columns
white: 4898 rows, 12 columns


In [71]:
red.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [72]:
# Column types: all should be numeric (float64, quality int64)
pd.DataFrame({"red": red.dtypes, "white": white.dtypes})

,red,white
fixed acidity,float64,float64
volatile acidity,float64,float64
citric acid,float64,float64
residual sugar,float64,float64
chlorides,float64,float64
free sulfur dioxide,float64,float64
total sulfur dioxide,float64,float64
density,float64,float64
pH,float64,float64
sulphates,float64,float64


### 1. Feature types

In [73]:
both = pd.concat([red, white])
feature_info = pd.DataFrame({
    "dtype": both.dtypes.astype(str),
    "n_unique": both.nunique(),
    "min": both.min(),
    "max": both.max(),
    "all_whole_numbers": (both % 1 == 0).all(),
})
feature_info

,dtype,n_unique,min,max,all_whole_numbers
fixed acidity,float64,106,3.80000,15.90000,False
volatile acidity,float64,187,0.08000,1.58000,False
citric acid,float64,89,0.00000,1.66000,False
residual sugar,float64,316,0.60000,65.80000,False
chlorides,float64,214,0.00900,0.61100,False
free sulfur dioxide,float64,135,1.00000,289.00000,False
total sulfur dioxide,float64,276,6.00000,440.00000,False
density,float64,998,0.98711,1.03898,False
pH,float64,108,2.72000,4.01000,False
sulphates,float64,111,0.22000,2.00000,False


Classification (based on the UCI dataset page and Cortez et al. 2009):

| Feature | Quantitative / Qualitative | Numerical / Categorical | Discrete / Continuous |
|---|---|---|---|
| `fixed acidity`, `volatile acidity`, `citric acid`, `residual sugar`, `chlorides`, `free sulfur dioxide`, `total sulfur dioxide`, `density`, `pH`, `sulphates`, `alcohol` | Quantitative | Numerical | Continuous |
| `quality` (label, score 0–10) | Qualitative (ordinal) | Categorical | Discrete |
| `type` (added at integration: red / white) | Qualitative (nominal) | Categorical | Discrete |

Note: `quality` is the median of expert grades on an ordered scale: the order matters, but the distance between two grades is not a measured quantity.

### 2. Missing values

In [74]:
missing = pd.DataFrame({
    "red_missing": red.isna().sum(),
    "white_missing": white.isna().sum(),
})
print("Rows with at least one missing value:")
for name, df in datasets.items():
    print(f"  {name}: {df.isna().any(axis=1).sum()}")
missing

Rows with at least one missing value:
  red: 0
  white: 0


,red_missing,white_missing
fixed acidity,0,0
volatile acidity,0,0
citric acid,0,0
residual sugar,0,0
chlorides,0,0
free sulfur dioxide,0,0
total sulfur dioxide,0,0
density,0,0
pH,0,0
sulphates,0,0


### 3. Duplicate records

In [75]:
for name, df in datasets.items():
    n_dup = df.duplicated().sum()
    print(f"{name}: {n_dup} duplicate rows ({n_dup / len(df) * 100:.2f}%)")

red: 240 duplicate rows (15.01%)
white: 937 duplicate rows (19.13%)


In [76]:
# Look at some duplicated groups (keep=False shows all copies)
red[red.duplicated(keep=False)].sort_values(list(red.columns)).head(6)

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
142,5.2,0.34,0.00,1.8,0.050,27.0,63.0,0.9916,3.68,0.79,14.0,6
144,5.2,0.34,0.00,1.8,0.050,27.0,63.0,0.9916,3.68,0.79,14.0,6
131,5.6,0.50,0.09,2.3,0.049,17.0,99.0,0.9937,3.63,0.63,13.0,5
132,5.6,0.50,0.09,2.3,0.049,17.0,99.0,0.9937,3.63,0.63,13.0,5
1488,5.6,0.54,0.04,1.7,0.049,5.0,13.0,0.9942,3.72,0.58,11.4,5
1491,5.6,0.54,0.04,1.7,0.049,5.0,13.0,0.9942,3.72,0.58,11.4,5


### 4. Inconsistency
Rows with the same 11 features but a different `quality`

In [77]:
features = [c for c in red.columns if c != "quality"]

for name, df in datasets.items():
    same_features = df.duplicated(subset=features, keep=False)
    groups = df[same_features].groupby(features)["quality"].nunique()
    n_conflict = (groups > 1).sum()
    print(f"{name}: {(groups > 0).sum()} duplicate feature groups, "
          f"{n_conflict} with different quality")

red: 220 duplicate feature groups, 0 with different quality
white: 772 duplicate feature groups, 0 with different quality


### Drop duplicates

In [78]:
red_clean = red.drop_duplicates().reset_index(drop=True)
white_clean = white.drop_duplicates().reset_index(drop=True)
datasets_clean = {"red": red_clean, "white": white_clean}

for name in datasets:
    before, after = len(datasets[name]), len(datasets_clean[name])
    print(f"{name}: {before} -> {after} rows ({before - after} removed, "
          f"{(before - after) / before * 100:.2f}%)")

red: 1599 -> 1359 rows (240 removed, 15.01%)
white: 4898 -> 3961 rows (937 removed, 19.13%)


## Summary before cleaning

In [79]:
features = [c for c in red.columns if c != "quality"]

def n_inconsistent(df):
    q_per_group = df.groupby(features)["quality"].transform("nunique")
    return int((q_per_group > 1).sum())

summary_before = pd.DataFrame({
    name: {
        "Total": len(df),
        "Missing value": int(df.isna().any(axis=1).sum()),
        "Duplicate": int(df.duplicated().sum()),
        "Inconsistency": n_inconsistent(df),
    }
    for name, df in datasets.items()
})
summary_before

,red,white
Total,1599,4898
Missing value,0,0
Duplicate,240,937
Inconsistency,0,0


### 5. Incorrect values

In [80]:
# --- Incorrect values (checked on duplicate-free data) ---
features = [c for c in red_clean.columns if c != 'quality']

def check_incorrect(df, name):
    rules = {
        'Negative value (any feature)': (df[features] < 0).any(axis=1),
        'pH outside 0-14': ~df['pH'].between(0, 14),
        'Alcohol outside 0-100 % vol': ~df['alcohol'].between(0, 100),
        'Density <= 0': df['density'] <= 0,
        'Free SO2 > total SO2': df['free sulfur dioxide'] > df['total sulfur dioxide'],
        'Quality not an integer 0-10': ~df['quality'].between(0, 10) | (df['quality'] % 1 != 0),
    }
    result = pd.DataFrame({'Rows flagged': {k: int(v.sum()) for k, v in rules.items()}})
    result['Percentage (%)'] = (result['Rows flagged'] / len(df) * 100).round(2)
    any_flag = pd.concat(rules.values(), axis=1).any(axis=1)
    print(f'{name}: {int(any_flag.sum())} of {len(df)} rows have at least one incorrect value')
    return result, any_flag

res_red, bad_red = check_incorrect(red_clean, 'Red')
res_white, bad_white = check_incorrect(white_clean, 'White')
display(pd.concat({'Red': res_red, 'White': res_white}, axis=1))

Red: 0 of 1359 rows have at least one incorrect value
White: 0 of 3961 rows have at least one incorrect value


Red                       White  \
                             Rows flagged Percentage (%) Rows flagged   
Negative value (any feature)            0            0.0            0   
pH outside 0-14                         0            0.0            0   
Alcohol outside 0-100 % vol             0            0.0            0   
Density <= 0                            0            0.0            0   
Free SO2 > total SO2                    0            0.0            0   
Quality not an integer 0-10             0            0.0            0   

                                             
                             Percentage (%)  
Negative value (any feature)            0.0  
pH outside 0-14                         0.0  
Alcohol outside 0-100 % vol             0.0  
Density <= 0                            0.0  
Free SO2 > total SO2                    0.0  
Quality not an integer 0-10             0.0

In [81]:
display(pd.concat({'Red': red_clean[features].agg(['min', 'max']).T,
                   'White': white_clean[features].agg(['min', 'max']).T}, axis=1))

Red               White           
                          min        max      min        max
fixed acidity         4.60000   15.90000  3.80000   14.20000
volatile acidity      0.12000    1.58000  0.08000    1.10000
citric acid           0.00000    1.00000  0.00000    1.66000
residual sugar        0.90000   15.50000  0.60000   65.80000
chlorides             0.01200    0.61100  0.00900    0.34600
free sulfur dioxide   1.00000   72.00000  2.00000  289.00000
total sulfur dioxide  6.00000  289.00000  9.00000  440.00000
density               0.99007    1.00369  0.98711    1.03898
pH                    2.74000    4.01000  2.72000    3.82000
sulphates             0.33000    2.00000  0.22000    1.08000
alcohol               8.40000   14.90000  8.00000   14.20000